In [ ]:
#pip install mlflow
#MLflow model ve deney yönetimi için kullanılan açık kaynaklı bir platformdur.
#Makine öğrenimi projelerinde model izleme, yeniden üretilebilirlik ve dağıtım süreçlerini kolaylaştırır.



In [31]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

# 1. Veri Setini Yükle (Sklearn'den)
cancer = load_breast_cancer()
cancer_df = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)
#mlflow kullanımı için örnek bir data seti sklearn kütüphanesinden yüklendi.


In [32]:
cancer_df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [33]:
cancer_df.describe().T

,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


In [34]:
X = cancer.data
y = cancer.target
type(X)

numpy.ndarray

In [35]:
print(y)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 1 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 0 0 1 0 0 1 1 1 1 0 1 0 0 1 1 1 1 0 1 0 0
 1 0 1 0 0 1 1 1 0 0 1 0 0 0 1 1 1 0 1 1 0 0 1 1 1 0 0 1 1 1 1 0 1 1 0 1 1
 1 1 1 1 1 1 0 0 0 1 0 0 1 1 1 0 0 1 0 1 0 0 1 0 0 1 1 0 1 1 0 1 1 1 1 0 1
 1 1 1 1 1 1 1 1 0 1 1 1 1 0 0 1 0 1 1 0 0 1 1 0 0 1 1 1 1 0 1 1 0 0 0 1 0
 1 0 1 1 1 0 1 1 0 0 1 0 0 0 0 1 0 0 0 1 0 1 0 1 1 0 1 0 0 0 0 1 1 0 0 1 1
 1 0 1 1 1 1 1 0 0 1 1 0 1 1 0 0 1 0 1 1 1 1 0 1 1 1 1 1 0 1 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 1 1 1 1 1 1 0 1 0 1 1 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1 1 1 1 1
 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 0 1 0 1 1 1 1 0 0 0 1 1
 1 1 0 1 0 1 0 1 1 1 0 1 1 1 1 1 1 1 0 0 0 1 1 1 1 1 1 1 1 1 1 1 0 0 1 0 0
 0 1 0 0 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1
 1 0 1 1 1 1 1 0 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 0 1 0 0 1 0 1 1 1 1 1 0 1 1
 0 1 0 1 1 0 1 0 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 0 1
 1 1 1 1 1 1 0 1 0 1 1 0 

In [36]:
print(X)

[[1.799e+01 1.038e+01 1.228e+02 ... 2.654e-01 4.601e-01 1.189e-01]
 [2.057e+01 1.777e+01 1.329e+02 ... 1.860e-01 2.750e-01 8.902e-02]
 [1.969e+01 2.125e+01 1.300e+02 ... 2.430e-01 3.613e-01 8.758e-02]
 ...
 [1.660e+01 2.808e+01 1.083e+02 ... 1.418e-01 2.218e-01 7.820e-02]
 [2.060e+01 2.933e+01 1.401e+02 ... 2.650e-01 4.087e-01 1.240e-01]
 [7.760e+00 2.454e+01 4.792e+01 ... 0.000e+00 2.871e-01 7.039e-02]]


In [37]:

# Eğitim ve Test olarak ayır
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [38]:

# 2. Deney Adı Belirle
mlflow.set_experiment("Kanser_Teshis_Projesi")


<Experiment: artifact_location='/home/guckan/Desktop/ornekveriproje/mlruns/1', creation_time=1765126753669, experiment_id='1', last_update_time=1765126753669, lifecycle_stage='active', name='Kanser_Teshis_Projesi', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [39]:
# Farklı 'n_estimators' (ağaç sayısı) değerlerini deniyoruz: 10 ve 50
for n_agac in [10, 50]:
    
    # Her deneme için yeni bir 'Run' başlat
    with mlflow.start_run(run_name=f"RandomForest_n{n_agac}"):
        
        # A. Parametreleri Logla
        mlflow.log_param("n_estimators", n_agac)
        mlflow.log_param("criterion", "gini")
        
        # Modeli Eğit
        clf = RandomForestClassifier(n_estimators=n_agac, random_state=42)
        clf.fit(X_train, y_train)
        
        # Tahmin Yap
        y_pred = clf.predict(X_test)
        
        # B. Metrikleri Hesapla ve Logla
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        
        # C. Modeli Kaydet
        mlflow.sklearn.log_model(clf, "model")
        
        print(f"Deneme Bitti -> Ağaç Sayısı: {n_agac}, Accuracy: {acc:.4f}")

print("\n--- Tüm denemeler tamamlandı. MLflow UI'dan sonuçları kıyaslayabilirsin. ---")

2025/12/07 20:28:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/07 20:28:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Deneme Bitti -> Ağaç Sayısı: 10, Accuracy: 0.9561
Deneme Bitti -> Ağaç Sayısı: 50, Accuracy: 0.9649

--- Tüm denemeler tamamlandı. MLflow UI'dan sonuçları kıyaslayabilirsin. ---


In [40]:
#conda activate base
#mlflow ui
#5000 portundan mlflow arayüzüne erişebilirsin.
#http://localhost:5000


In [41]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

In [42]:


#Farklı modelleri bir sözlükte tanımlayalım
modeller = {
    "Lojistik_Regresyon": LogisticRegression(max_iter=1000),
    "Karar_Agaci": DecisionTreeClassifier(max_depth=5),
    "Random_Forest": RandomForestClassifier(n_estimators=50),
    "SVM_Modeli": SVC(),
    "KNN_Modeli": KNeighborsClassifier(n_neighbors=5)
}


In [43]:

mlflow.set_experiment("Farkli_Modeller")


<Experiment: artifact_location='/home/guckan/Desktop/ornekveriproje/mlruns/3', creation_time=1765127951544, experiment_id='3', last_update_time=1765127951544, lifecycle_stage='active', name='Farkli_Modeller', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [44]:
#Döngü ile Her Modeli Eğit ve Kaydet
from sklearn.metrics import precision_score, recall_score, accuracy_score


for model_adi, model_nesnesi in modeller.items():
    
    with mlflow.start_run(run_name=model_adi):
        
        # A. Modeli Eğit
        model_nesnesi.fit(X_train, y_train)
        preds = model_nesnesi.predict(X_test)
        
        # B. Metrikleri Hesapla
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        recall = recall_score(y_test, preds)
        precision = precision_score(y_test, preds)
        accuracy = accuracy_score(y_test, preds)

        
        # C. MLflow'a Kaydet
        # Algoritma ismini parametre olarak ekliyoruz ki filtrelemesi kolay olsun
        mlflow.log_param("algoritma", model_adi)
        
        # Modelin kendi parametrelerini de (varsayılan olsa bile) kaydedebiliriz
        mlflow.log_params(model_nesnesi.get_params())
        
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("accuracy_score", accuracy)   
        
        mlflow.sklearn.log_model(model_nesnesi, "model")
        
        print(f"✅ {model_adi} tamamlandı. Accuracy: {acc:.4f}")

print("\n--- Tüm modeller MLflow'a gönderildi ---")

/home/guckan/Downloads/yes/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2025/12/07 20:28:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/12/07 20:28:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Lojistik_Regresyon tamamlandı. Accuracy: 0.9649


2025/12/07 20:28:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Karar_Agaci tamamlandı. Accuracy: 0.9386


2025/12/07 20:28:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Random_Forest tamamlandı. Accuracy: 0.9649


2025/12/07 20:28:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ SVM_Modeli tamamlandı. Accuracy: 0.9474
✅ KNN_Modeli tamamlandı. Accuracy: 0.9561

--- Tüm modeller MLflow'a gönderildi ---


In [45]:
#conda activate base (eğer conda kullanıyorsan ilgli ortamı terminalden aktifleştir)
#mlflow ui
#5000 portundan mlflow arayüzüne erişebilirsin.
#http://localhost:5000
